In [1]:
import pandas as pd
from surprise import Reader, Dataset, SVD, accuracy
from surprise.model_selection import train_test_split


movies_df = pd.read_csv(
    "../data/movies.dat",
    sep="::",
    engine="python",
    encoding="latin-1",
    names=["movie_id", "title", "genres"]
)

ratings_df = pd.read_csv(
    "../data/ratings.dat",
    sep="::",
    engine="python",
    encoding="latin-1",
    names=["user_id", "movie_id", "rating", "timestamp"]
)
print(f"Loaded {len(ratings_df)} ratings and {len(movies_df)} movies.")

Loaded 1000209 ratings and 3883 movies.


In [2]:
# Drop timestamp as it is not needed for basic SVD
ratings_df = ratings_df.drop(columns=['timestamp'], errors='ignore')

# Define the rating scale (1 to 5) for the Reader
reader = Reader(rating_scale=(1, 5))

# Load the pandas DataFrame directly into a Surprise Dataset
data = Dataset.load_from_df(ratings_df[['user_id', 'movie_id', 'rating']], reader)

In [3]:
# Train on 90% of the data and validate on 10%
trainset, testset = train_test_split(data, test_size=0.10, random_state=42)

# Initialize the SVD algorithm
model = SVD(n_factors=50, random_state=42)

print("Training the SVD model...")
model.fit(trainset)
print("Training complete.")

Training the SVD model...
Training complete.


In [4]:
# Predict ratings for the validation set
predictions = model.test(testset)

# Compute and print evaluation metrics
print("Model Evaluation:")
accuracy.rmse(predictions)
accuracy.mae(predictions)

Model Evaluation:
RMSE: 0.8613
MAE:  0.6753


np.float64(0.6753371125872389)

In [5]:
# Let us get a user and see the top recommendations.
user_id = ratings_df['user_id'].sample(1, random_state=42).iloc[0]

# 1. Find movies the user has already watched
movies_watched_by_user = ratings_df[ratings_df['user_id'] == user_id]['movie_id'].tolist()

# 2. Identify movies the user has NOT watched
all_movies = movies_df['movie_id'].unique()
movies_not_watched = [m for m in all_movies if m not in movies_watched_by_user]

# 3. Predict the rating for all unwatched movies
# The predict method returns a Prediction object containing the estimated rating (est)
predictions = [model.predict(user_id, movie_id) for movie_id in movies_not_watched]

# 4. Sort predictions by the estimated rating in descending order and get the top 10
predictions.sort(key=lambda x: x.est, reverse=True)
top_10_predictions = predictions[:10]
recommended_movie_ids = [pred.iid for pred in top_10_predictions]

# --- Output Formatting ---
print("Showing recommendations for user: {}".format(user_id))
print("====" * 9)
print("Movies with high ratings from user")
print("----" * 8)

top_movies_user = ratings_df[ratings_df['user_id'] == user_id].sort_values(by="rating", ascending=False).head(5)['movie_id'].values
movie_df_rows = movies_df[movies_df["movie_id"].isin(top_movies_user)]
for row in movie_df_rows.itertuples():
    print(row.title, ":", row.genres)

print("----" * 8)
print("Top 10 movie recommendations")
print("----" * 8)

# Fetch movie details keeping the sorted order of recommendations
recommended_movies = movies_df.set_index('movie_id').loc[recommended_movie_ids]
for title, genres in zip(recommended_movies['title'], recommended_movies['genres']):
    print(title, ":", genres)

Showing recommendations for user: 5412
Movies with high ratings from user
--------------------------------
Apollo 13 (1995) : Drama
Schindler's List (1993) : Drama|War
Reservoir Dogs (1992) : Crime|Thriller
American Pie (1999) : Comedy
Big Trouble in Little China (1986) : Action|Comedy
--------------------------------
Top 10 movie recommendations
--------------------------------
Usual Suspects, The (1995) : Crime|Thriller
Close Shave, A (1995) : Animation|Comedy|Thriller
Shawshank Redemption, The (1994) : Drama
Swingers (1996) : Comedy|Drama
Wrong Trousers, The (1993) : Animation|Comedy
Sunset Blvd. (a.k.a. Sunset Boulevard) (1950) : Film-Noir
Wallace & Gromit: The Best of Aardman Animation (1996) : Animation
American History X (1998) : Drama
Sanjuro (1962) : Action|Adventure
Great Escape, The (1963) : Adventure|War


In [6]:
from surprise import dump
dump.dump('svd_model.pkl', algo=model)